# ThemeCloner V2 — revised point in time walk forward

This notebook keeps the existing data pull and RP PCA modules intact. The revised logic is isolated in three additive modules:

- `src/matching_v3.py`: robust multi ETF factor exposure matching
- `src/walkforward_v3.py`: point in time residualization, RP PCA fitting and basket formation
- `src/evaluation_v3.py`: forward exposure, placebo and semantic validation

The primary objective is to identify small cap stocks with related latent thematic exposures. Raw ETF return tracking is retained only as a secondary diagnostic.

In [1]:
# -------------------- 0. Setup --------------------
import os
import sys

ROOT = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(ROOT, "config", "etfs.csv")):
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print(f"root: {ROOT}")
print(f"config exists: {os.path.exists('config/etfs.csv')}")

root: C:\Users\aamin\ThemeCloner2
config exists: True


## Step 1 — Pull the same raw inputs used by V2

In [2]:
# -------------------- 1a. Parameters and ETF config --------------------
from src.data_pull import load_etf_config, split_themes_controls

START_DATE = "2018-01-01"
K = 15
GAMMA = 10.0
TOP_N = 30
TOP_FACTORS = 5                      # retain more of each theme's factor shape

REBALANCE_FREQ = "Q"
WALKFORWARD_WINDOW = "rolling"      # "rolling" or "expanding"
ROLLING_WINDOW_WEEKS = 208          # four years
MIN_TRAIN_OBS = 104                 # at least two years of valid observations
MIN_TRAIN_COVERAGE = 0.80

INCLUDE_COMMODITY = False
N_PLACEBOS = 20                     # use 100 to 200 for final reported results
PLACEBO_PVALUE_MAX = 0.10           # rank matched null admission threshold
PLACEBO_USE_FDR = False             # q values are reported; p values gate admission
RANDOM_STATE = 42

# Process workers are used for placebo batches. Half the logical CPUs avoids
# oversubscription on mixed performance and efficiency core laptops.
N_JOBS = min(8, max(1, (os.cpu_count() or 4) // 2))
print(f"Placebo process workers: {N_JOBS}")

etf_config = load_etf_config("config/etfs.csv")
themes_config, controls_config = split_themes_controls(etf_config)
print(f"{themes_config['theme'].nunique()} discovery themes")


Placebo process workers: 8
loaded 44 ETFs: 31 theme-ETFs, 13 control-ETFs (sector/subsector calibration anchors)
  AI Infrastructure: IGPT (IGPT), QTUM (QTUM), SOXX (SOXX)
  Agribusiness: MOO (MOO), VEGI (VEGI), PBJ (PBJ)
  Clean Energy: ICLN (ICLN), QCLN (QCLN), PBW (PBW)
  Critical Minerals: URA (URA), LIT (LIT), COPX (COPX)
  Cybersecurity: CIBR (CIBR), HACK (HACK), BUG (BUG)
  Defense: ITA (ITA), PPA (PPA), XAR (XAR)
  Factor1: IUSG (IUSG)
  Factor2: VLUE (VLUE)
  Factor3: QUAL (QUAL)
  Factor4: SPGP (SPGP)
  Factor5: MTUM (MTUM)
  Factor6: VUG (VUG)
  Factor7: VTV (VTV)
  Fintech: FINX (FINX), ARKF (ARKF), IPAY (IPAY)
  Robotics & AI: BOTZ (BOTZ), ROBO (ROBO), IRBO (IRBO)
  SECTOR: US Energy: XLE (XLE)
  SECTOR: US Financials: XLF (XLF)
  SECTOR: US Technology: XLK (XLK)
  SUBSECTOR: US Banks: KRE (KRE)
  SUBSECTOR: US Biotech: XBI (XBI)
  SUBSECTOR: US Software: IGV (IGV)
  Timber & Forestry: WOOD (WOOD), CUT (CUT)
  US Infrastructure: PAVE (PAVE), IFRA (IFRA)
  Water: PHO (PHO),

In [3]:
# -------------------- 1b. ETF returns --------------------
from src.data_pull import pull_etf_returns

etf_returns = pull_etf_returns(etf_config, start=START_DATE)
print(etf_returns.shape)


pulling ETF panel: 44 tickers in 1 batch(es)
  batch 1/1 (44 tickers)...
  dropped 1 tickers below 85% coverage
  ETF panel: 445 weeks x 43 stocks
  date range: 2018-01-12 to 2026-07-17
  ETFs kept: ['ARKF', 'BOTZ', 'CGW', 'CIBR', 'COPX', 'CUT', 'FINX', 'FIW', 'HACK', 'ICLN', 'IFRA', 'IGPT', 'IGV', 'IPAY', 'IRBO', 'ITA', 'IUSG', 'KRE', 'LIT', 'MOO', 'MTUM', 'PAVE', 'PBJ', 'PBW', 'PHO', 'PPA', 'QCLN', 'QTUM', 'QUAL', 'ROBO', 'SOXX', 'SPGP', 'URA', 'VEGI', 'VLUE', 'VTV', 'VUG', 'WOOD', 'XAR', 'XBI', 'XLE', 'XLF', 'XLK']
(445, 43)


In [4]:
# -------------------- 1c. Covariance universe --------------------
from src.data_pull import pull_covariance_universe

cov_returns = pull_covariance_universe(
    start=START_DATE,
    use_cache=True,
    us_only=True,
)
print(cov_returns.shape)

loaded covariance universe from cache (us_only): 1003 tickers across ['us_sp500', 'us_mid_small']

pulling covariance universe: 1003 unique tickers

pulling covariance universe: 1003 tickers in 3 batch(es)
  batch 1/3 (400 tickers)...


$SATS: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)

1 Failed download:
['SATS']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)


  batch 2/3 (400 tickers)...


$BAYA: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)

1 Failed download:
['BAYA']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)


  batch 3/3 (203 tickers)...


$TBH: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)

1 Failed download:
['TBH']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)


  dropped 251 tickers below 85% coverage
  covariance universe: 445 weeks x 752 stocks
  date range: 2018-01-12 to 2026-07-17
(445, 752)


In [5]:
# -------------------- 1d. Target universe --------------------
from src.data_pull import _cached_returns, pull_target_universe

tgt_returns = _cached_returns(
    "target_universe_returns",
    pull_target_universe,
    refresh=False,
    start=START_DATE,
)
print(tgt_returns.shape)

  loaded target_universe_returns from cache: 444 weeks x 811 cols
(444, 811)


In [6]:
# -------------------- 1e. Observable factor returns --------------------
# IMPORTANT: do not residualize the full sample here.
# The revised walk forward module fits residualization inside each rebalance window.
from src.residualize import get_combined_factors

ff_factors = get_combined_factors(
    start=START_DATE,
    include_commodity=INCLUDE_COMMODITY,
)
print(ff_factors.shape)


pulling FF5 + momentum from Kenneth French library (weekly)...
  momentum fetch failed via pandas_datareader (Unknown datetime string format, unable to parse: Missing data are indicated by -99.99 or -999., at position 0)
  falling back to direct download from Ken French's data library...
  first 20 raw lines (for diagnosis):
    'This file was created by using the 202605 CRSP database.  It,,'
    'contains a momentum factor, constructed from six value-weight portfolios formed,'
    'using independent sorts on size and prior return of NYSE, AMEX, and NASDAQ stocks.'
    'MOM is the average of the returns on two (big and small) high prior return portfolios,,'
    'minus the average of the returns on two low prior return portfolios.  The portfolios,,'
    'are constructed daily.  Big means a firm is above the median market cap on the NYSE,,'
    'at the end of the previous day; small firms are below the median NYSE market cap.,,'
    'Prior return is measured from day - 250 to - 21.  Fir

## Step 2 — Build the point in time rebalance schedule

In [7]:
# -------------------- 2. Rebalance dates --------------------
from src.rppca_walkforward import make_rebalance_dates

# Restrict the schedule to dates where the observable factor panel is available.
model_dates = cov_returns.index.intersection(ff_factors.index)
minimum_window = ROLLING_WINDOW_WEEKS if WALKFORWARD_WINDOW == "rolling" else MIN_TRAIN_OBS

rebalance_dates = make_rebalance_dates(
    model_dates,
    freq=REBALANCE_FREQ,
    min_window_weeks=minimum_window,
)
print(f"{len(rebalance_dates)} rebalance dates")
print(rebalance_dates[:3], "...", rebalance_dates[-3:])

18 rebalance dates
[Timestamp('2021-12-31 00:00:00'), Timestamp('2022-03-25 00:00:00'), Timestamp('2022-06-24 00:00:00')] ... [Timestamp('2025-09-26 00:00:00'), Timestamp('2025-12-26 00:00:00'), Timestamp('2026-03-27 00:00:00')]


## Step 3 — Matching rules

In [8]:
# -------------------- 3. Robust matching configuration --------------------
from src.matching_v3 import MatchingConfig

matching_config = MatchingConfig(
    top_factors=TOP_FACTORS,

    # Confidence filters use both raw and adjusted R squared.
    min_etf_r2=0.10,
    min_etf_adjusted_r2=0.05,
    min_candidate_r2=0.15,
    min_candidate_adjusted_r2=0.10,

    # Consensus checks across the theme ETF reference set.
    min_cosine=0.30,
    cosine_quantile=0.25,            # lower quartile cosine must clear the floor
    reference_distance_quantile=0.75,# poor ETF match cannot be averaged away

    max_relative_distance=None,
    factor_gap_weight=0.25,
    consensus_weight=0.25,

    min_etf_matches=1,
    min_etf_match_fraction=0.60,     # 2 of 2 ETFs; at least 2 of 3 ETFs
    etf_match_distance=None,

    # Overlap is diagnosed, not prohibited.
    max_theme_rank=None,
)
matching_config


MatchingConfig(top_factors=5, min_etf_r2=0.1, min_etf_adjusted_r2=0.05, min_candidate_r2=0.15, min_candidate_adjusted_r2=0.1, min_cosine=0.3, cosine_quantile=0.25, reference_distance_quantile=0.75, max_relative_distance=None, factor_gap_weight=0.25, consensus_weight=0.25, min_etf_matches=1, min_etf_match_fraction=0.6, etf_match_distance=None, max_theme_rank=None)

## Step 4 — Run the revised walk forward process

### 3B. Parallel execution

Ticker regressions are vectorized. Independent placebo batches use separate process workers, so the earlier Windows `threadpoolctl` workaround is no longer required.


In [9]:
# -------------------- 4. Point in time backtest --------------------
from src.walkforward_v3 import run_point_in_time_backtest

results = run_point_in_time_backtest(
    cov_returns_raw=cov_returns,
    etf_returns_raw=etf_returns,
    target_returns_raw=tgt_returns,
    factor_returns=ff_factors,
    etf_config=themes_config,
    rebalance_dates=rebalance_dates,
    K=K,
    gamma=GAMMA,
    window=WALKFORWARD_WINDOW,
    rolling_window_weeks=ROLLING_WINDOW_WEEKS,
    top_n=TOP_N,
    matching_config=matching_config,
    min_train_obs=MIN_TRAIN_OBS,
    min_train_coverage=MIN_TRAIN_COVERAGE,
    n_placebos=N_PLACEBOS,
    placebo_pvalue_max=PLACEBO_PVALUE_MAX,
    placebo_use_fdr=PLACEBO_USE_FDR,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=True,
)



Rebalance 1/18: 2021-12-31
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.050
  top eigenvalues:   [61.8 41.7 25.2 21.2 17.4 16.7 14.9 13.  12.5 11.4 10.2  9.4  9.2  8.8
  8.7]
Selected 0 positions across 11 themes; forward window has 12 observations.
Timing seconds: residualization=0.2, rppca=0.9, reference_and_candidate_fit=5.2, placebos_and_admission=27.9, forward_evaluation=1.6, placebo_forward_metrics=0.6, total=36.3

Rebalance 2/18: 2022-03-25
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      6.540
  top eigenvalues:   [63.7 40.5 25.4 22.6 18.3 16.3 14.8 13.5 12.5 11.4 10.4  9.6  9.   9.
  8.7]
Selected 0 positions across 11 themes; forward window has 13 observations.
Timing seconds: residualization=0.2, rppca=1.0, reference_and_candidate_fit=4.3, placebos_and_admission=17.4, forward_evaluation=0.6, placebo_forward_metrics=0.5, total=24.0

Rebalance 3/18: 2022-06-24
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      6.446
  top eigenvalues:   [62.8 35.6 26.3 22.  18

RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.649
  top eigenvalues:   [62.4 49.5 26.4 20.6 19.7 15.5 14.8 13.7 11.5 11.1 10.2  9.5  9.3  9.2
  8.8]
Selected 0 positions across 11 themes; forward window has 13 observations.
Timing seconds: residualization=0.2, rppca=1.3, reference_and_candidate_fit=6.9, placebos_and_admission=22.2, forward_evaluation=0.6, placebo_forward_metrics=0.5, total=31.8

Rebalance 16/18: 2025-09-26
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.023
  top eigenvalues:   [64.7 47.5 26.  20.3 19.  16.6 14.1 13.8 11.6 11.3 10.2  9.7  9.3  8.9
  8.7]
Selected 0 positions across 11 themes; forward window has 13 observations.
Timing seconds: residualization=0.2, rppca=1.0, reference_and_candidate_fit=6.0, placebos_and_admission=19.0, forward_evaluation=0.6, placebo_forward_metrics=0.3, total=27.1

Rebalance 17/18: 2025-12-26
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.298
  top eigenvalues:   [68.1 46.  26.6 20.3 18.8 16.9 14.3 12.7 12.  11

## Step 5 — Review the latest selected baskets

In [12]:
# -------------------- 5a. Latest baskets --------------------
from src.matching_v3 import flatten_baskets

basket_history = flatten_baskets(results["period_baskets"])

if basket_history.empty:
    print(
        "No stocks passed the current matching thresholds in any rebalance period."
    )

    # Show how many names passed in each period and theme.
    selection_counts = pd.DataFrame(
        [
            {
                "rebalance_date": date,
                "theme": theme,
                "n_selected": len(basket.get("tickers", [])),
            }
            for date, theme_map in results["period_baskets"].items()
            for theme, basket in theme_map.items()
        ]
    )

    display(selection_counts)
else:
    latest_date = basket_history["rebalance_date"].max()
    latest_baskets = basket_history.loc[
        basket_history["rebalance_date"] == latest_date
    ]

    print(f"Latest rebalance: {latest_date.date()}")
    display(
        latest_baskets.sort_values(["theme", "ticker"])
    )

No stocks passed the current matching thresholds in any rebalance period.


,rebalance_date,theme,n_selected
0,2021-12-31,AI Infrastructure,0
1,2021-12-31,Agribusiness,0
2,2021-12-31,Clean Energy,0
3,2021-12-31,Critical Minerals,0
4,2021-12-31,Cybersecurity,0
...,...,...,...
193,2026-03-27,Fintech,0
194,2026-03-27,Robotics & AI,0
195,2026-03-27,Timber & Forestry,0
196,2026-03-27,US Infrastructure,0


In [15]:
# -------------------- 5c. Rebuild baskets without placebo admission --------------------
from src.matching_v3 import select_equal_weight_baskets, flatten_baskets

period_baskets_base = {}

for date, scores in results["period_scores"].items():
    revised_scores = scores.copy()
    revised_scores["eligible"] = revised_scores["base_eligible"]

    period_baskets_base[date] = select_equal_weight_baskets(
        revised_scores,
        top_n=TOP_N,
    )

basket_history_base = flatten_baskets(period_baskets_base)

latest_date = basket_history_base["rebalance_date"].max()
latest_baskets_base = basket_history_base.loc[
    basket_history_base["rebalance_date"] == latest_date
]

print(f"Latest rebalance: {latest_date.date()}")
print(f"Selected positions: {len(latest_baskets_base)}")
print(f"Unique stocks: {latest_baskets_base['ticker'].nunique()}")

display(
    latest_baskets_base.sort_values(["theme", "ticker"])
)

Latest rebalance: 2026-03-27
Selected positions: 154
Unique stocks: 99


,rebalance_date,theme,ticker,weight
2733,2026-03-27,AI Infrastructure,NVEC,0.333333
2734,2026-03-27,AI Infrastructure,PAMT,0.333333
2732,2026-03-27,AI Infrastructure,PFX,0.333333
2737,2026-03-27,Agribusiness,ACNT,0.100000
2739,2026-03-27,Agribusiness,ASA,0.100000
...,...,...,...,...
2881,2026-03-27,US Infrastructure,MTUS,0.333333
2883,2026-03-27,US Infrastructure,NGVC,0.333333
2882,2026-03-27,US Infrastructure,SXC,0.333333
2885,2026-03-27,Water,GWRS,0.500000


In [19]:
# -------------------- 5b. Latest detailed scores --------------------

latest_scores = results["period_scores"][latest_date]

latest_scores.loc[
    latest_scores["base_eligible"]
].sort_values(
    ["theme", "penalized_distance"]
).groupby(
    "theme",
    sort=False,
).head(10)[
    [
        "theme",
        "ticker",
        "penalized_distance",
        "consensus_cosine",
        "candidate_r2",
        "candidate_adjusted_r2",
        "n_etf_matches",
        "required_etf_matches",
        "placebo_rank_pvalue",
        "theme_distance_rank",
        "n_eligible_themes",
    ]
]

,theme,ticker,penalized_distance,consensus_cosine,candidate_r2,candidate_adjusted_r2,n_etf_matches,required_etf_matches,placebo_rank_pvalue,theme_distance_rank,n_eligible_themes
16,AI Infrastructure,PFX,1.775172,0.350626,0.167835,0.102822,3,2,0.500000,9.0,0
46,AI Infrastructure,NVEC,1.925454,0.470960,0.172289,0.107624,3,2,0.500000,3.0,0
323,AI Infrastructure,PAMT,2.733574,0.305538,0.184308,0.120582,2,2,0.500000,4.0,0
814,Agribusiness,PFX,1.342059,0.385212,0.167835,0.102822,3,2,0.142857,2.0,0
885,Agribusiness,HRZN,1.962728,0.300122,0.166420,0.101296,2,2,0.142857,5.0,0
...,...,...,...,...,...,...,...,...,...,...,...
7501,US Infrastructure,MTUS,2.373344,0.371088,0.195500,0.132648,2,2,0.200000,6.0,0
7589,US Infrastructure,SXC,2.630698,0.401059,0.256507,0.198422,2,2,0.200000,5.0,0
7707,US Infrastructure,NGVC,3.116400,0.309654,0.220963,0.160100,2,2,0.400000,3.0,0
8118,Water,PFX,1.487139,0.418874,0.167835,0.102822,3,2,0.166667,4.0,0


In [20]:
# -------------------- 5c. Selection overlap diagnostic --------------------

theme_counts = (
    latest_baskets_base.groupby("theme")
    .agg(
        n_selected=("ticker", "size"),
        n_unique=("ticker", "nunique"),
    )
    .sort_values("n_selected")
)

ticker_overlap = (
    latest_baskets_base.groupby("ticker")
    .agg(
        n_themes=("theme", "nunique"),
        themes=("theme", lambda x: ", ".join(sorted(set(x)))),
    )
    .query("n_themes > 1")
    .sort_values(["n_themes", "ticker"], ascending=[False, True])
)

display(theme_counts)
display(ticker_overlap.head(30))

,n_selected,n_unique
theme,,
Water,2,2
AI Infrastructure,3,3
US Infrastructure,3,3
Agribusiness,10,10
Defense,10,10
Fintech,12,12
Clean Energy,13,13
Robotics & AI,17,17
Critical Minerals,27,27


,n_themes,themes
ticker,,
GOOS,5,"Agribusiness, Clean Energy, Critical Minerals,..."
PFX,5,"AI Infrastructure, Agribusiness, Defense, Robo..."
ASA,4,"Agribusiness, Critical Minerals, Defense, Timb..."
MTUS,4,"Agribusiness, Critical Minerals, Timber & Fore..."
CSIQ,3,"Clean Energy, Critical Minerals, Robotics & AI"
ELME,3,"Cybersecurity, Fintech, Timber & Forestry"
GSM,3,"Agribusiness, Critical Minerals, Timber & Fore..."
JKS,3,"Clean Energy, Critical Minerals, Robotics & AI"
LGO,3,"Critical Minerals, Defense, Timber & Forestry"


In [21]:
# -------------------- 5d. Are the theme reference sets distinct? --------------------
latest_reference_distances = results["theme_reference_distances"]
latest_reference_distances = latest_reference_distances.loc[
    latest_reference_distances["rebalance_date"] == latest_date
]
latest_reference_distances.sort_values(
    ["median_relative_distance", "maximum_cosine"],
    ascending=[True, False],
).head(30)


,rebalance_date,theme_1,theme_2,median_relative_distance,minimum_relative_distance,maximum_cosine,median_cosine
982,2026-03-27,US Infrastructure,Water,0.794210,0.548402,0.721245,0.501960
959,2026-03-27,Clean Energy,Critical Minerals,0.845847,0.671332,0.648794,0.485091
935,2026-03-27,Robotics & AI,AI Infrastructure,0.894310,0.703614,0.715712,0.626235
945,2026-03-27,AI Infrastructure,Clean Energy,0.912352,0.771661,0.545223,0.431990
983,2026-03-27,US Infrastructure,Timber & Forestry,0.918566,0.616907,0.562512,0.421929
989,2026-03-27,Water,Timber & Forestry,0.947484,0.840975,0.300506,0.252004
988,2026-03-27,Critical Minerals,Timber & Forestry,0.977560,0.715519,0.733075,0.679096
970,2026-03-27,Defense,US Infrastructure,0.986977,0.909945,0.510208,0.396064
985,2026-03-27,Agribusiness,Water,0.998769,0.825632,0.429598,0.180585
986,2026-03-27,Agribusiness,Timber & Forestry,1.019678,0.792669,0.641059,0.462263


In [22]:
# -------------------- 5e. Runtime bottleneck diagnostic --------------------
results["timings"].sort_values("rebalance_date")


,rebalance_date,residualization,rppca,reference_and_candidate_fit,placebos_and_admission,forward_evaluation,placebo_forward_metrics,total
0,2021-12-31,0.159066,0.928541,5.153850,27.914362,1.612271,0.563121,36.331227
1,2022-03-25,0.154215,1.045589,4.299515,17.393040,0.596205,0.484866,23.973446
2,2022-06-24,0.169491,1.095447,4.431720,13.949762,0.778791,0.443900,20.869129
3,2022-09-30,0.133423,0.988558,4.914505,17.998007,0.528324,0.395052,24.957883
4,2022-12-30,0.144766,0.951972,4.647187,12.627120,0.859691,0.411599,19.642351
5,2023-03-31,0.176787,0.788857,4.501825,15.477798,0.764364,0.707260,22.416910
6,2023-06-30,0.160146,0.874941,4.290671,19.601829,0.570329,0.578901,26.076832
7,2023-09-29,0.160253,0.788817,4.668456,19.362377,0.557042,0.483149,26.020110
8,2023-12-29,0.154894,0.826613,4.636628,17.630579,0.637277,0.554659,24.440665
9,2024-03-29,0.126753,0.781001,4.665924,19.078138,0.875813,0.794375,26.322023


## Step 6 — Forward exposure evaluation

In [23]:
# -------------------- 6a. Theme level summary --------------------
from src.evaluation_v3 import summarize_forward_evaluation

evaluation_summary = summarize_forward_evaluation(results["evaluation"])
evaluation_summary.sort_values("median_rank_ic", ascending=False)

,theme,periods,median_rank_ic,mean_rank_ic,median_top_bottom_spread,median_selected_forward_corr,median_basket_etf_corr,median_exposure_hit_rate,median_placebo_pvalue,rank_ic_positive_rate
3,Cybersecurity,18,0.240073,0.172494,0.149500,NaN,NaN,NaN,NaN,0.777778
9,Water,18,0.167857,-0.005476,0.100368,NaN,NaN,NaN,NaN,0.625000
2,Clean Energy,18,0.091955,0.048570,0.106305,NaN,NaN,NaN,NaN,0.555556
10,Timber & Forestry,18,0.075375,0.077644,0.023134,NaN,NaN,NaN,NaN,0.611111
4,Defense,18,0.044012,0.040944,0.023658,NaN,NaN,NaN,NaN,0.666667
0,Robotics & AI,18,0.000000,0.017013,-0.024514,NaN,NaN,NaN,NaN,0.466667
5,Fintech,17,-0.050062,0.030945,0.011346,NaN,NaN,NaN,NaN,0.437500
1,AI Infrastructure,18,-0.100000,-0.044848,-0.155619,NaN,NaN,NaN,NaN,0.400000
8,Critical Minerals,18,-0.155606,-0.126292,-0.105581,NaN,NaN,NaN,NaN,0.388889
6,US Infrastructure,17,-0.214286,-0.108903,-0.243116,NaN,NaN,NaN,NaN,0.285714


In [24]:
# -------------------- 6b. Period level metrics --------------------
# Primary metrics:
# 1. forward_rank_ic_corr
# 2. top_bottom_corr_spread
# 3. basket_residual_etf_corr
# 4. placebo_pvalue
results["evaluation"].sort_values(["theme", "rebalance_date"])

,rebalance_date,theme,n_selected,n_base_eligible,forward_rank_ic_beta,forward_rank_ic_corr,top_bottom_beta_spread,top_bottom_corr_spread,selected_mean_forward_beta,selected_mean_forward_corr,exposure_hit_rate,basket_synthetic_beta,basket_synthetic_corr,basket_residual_etf_beta,basket_residual_etf_corr,raw_basket_period_return,n_benchmark_etfs,placebo_pvalue,placebo_median_basket_corr
1,2021-12-31,AI Infrastructure,0,9,-0.366667,-0.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,-0.167154
12,2022-03-25,AI Infrastructure,0,10,-0.115152,0.054545,-0.842833,-0.076545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,-0.234802
23,2022-06-24,AI Infrastructure,0,5,-0.200000,-0.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,0.095129
34,2022-09-30,AI Infrastructure,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,-0.349821
45,2022-12-30,AI Infrastructure,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150,2025-03-28,Water,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,0.235234
161,2025-06-27,Water,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,-0.166371
172,2025-09-26,Water,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,-0.165442
183,2025-12-26,Water,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,-0.347525


In [25]:
# -------------------- 6c. Rank matched placebo admission --------------------
# Each real rank is compared with the same rank from time shifted null themes.
results["placebo_rank_thresholds"].tail(30)


,rebalance_date,theme,rank,actual_penalized_distance,null_alpha_distance,placebo_rank_pvalue,n_placebo_ranks
2310,2026-03-27,Timber & Forestry,6,2.094108,2.725741,0.333333,2
2311,2026-03-27,Timber & Forestry,7,2.129409,2.869335,0.333333,2
2312,2026-03-27,Timber & Forestry,8,2.157661,3.080779,0.333333,2
2313,2026-03-27,Timber & Forestry,9,2.212302,3.328514,0.333333,2
2314,2026-03-27,Timber & Forestry,10,2.236160,3.378212,0.333333,2
2315,2026-03-27,Timber & Forestry,11,2.259279,3.551825,0.333333,2
2316,2026-03-27,Timber & Forestry,12,2.329720,3.616697,0.333333,2
2317,2026-03-27,Timber & Forestry,13,2.393388,3.744201,0.333333,2
2318,2026-03-27,Timber & Forestry,14,2.510492,3.860579,0.333333,2
2319,2026-03-27,Timber & Forestry,15,2.520089,4.043440,0.333333,2


## Step 7 — Independent economic relevance review

In [28]:
# -------------------- 7a. Create a blinded review sample --------------------
# Score each company from 0 to 3 using filings available before the rebalance date:
# 0 = no exposure, 1 = incidental, 2 = meaningful, 3 = central business driver.
import importlib
import src.evaluation_v3 as ev3


importlib.reload(ev3)
make_semantic_review_sample = ev3.make_semantic_review_sample

semantic_sample = make_semantic_review_sample(
    latest_scores,
    metadata=None,          # optionally pass ticker indexed sector and market cap data
    top_k=10,
    controls_per_candidate=1,
    random_state=RANDOM_STATE,
)

os.makedirs("outputs", exist_ok=True)
semantic_sample["review_sheet"].to_csv(
    "outputs/semantic_review_sheet.csv", index=False
)
semantic_sample["review_key"].to_csv(
    "outputs/semantic_review_key_PRIVATE.csv", index=False
)
semantic_sample["review_sheet"].head()

,review_id,theme,ticker,relevance_score,reviewer_notes
0,R0020,Agribusiness,BFST,NaN,
1,R0046,Clean Energy,EEX,NaN,
2,R0140,Robotics & AI,ACB,NaN,
3,R0031,Clean Energy,GOOS,NaN,
4,R0068,Cybersecurity,ASA,NaN,


In [30]:
# -------------------- 7b. Evaluate a completed review --------------------
# Run only after filling relevance_score in outputs/semantic_review_sheet.csv.
from src.evaluation_v3 import evaluate_semantic_review

completed_review = pd.read_csv("outputs/semantic_review_sheet.csv")
review_key = pd.read_csv("outputs/semantic_review_key_PRIVATE.csv")

n_completed = pd.to_numeric(
    completed_review["relevance_score"], errors="coerce"
).notna().sum()

if n_completed == 0:
    print(
        "No semantic reviews completed. "
        "Fill relevance_score before running this evaluation."
    )
else:
    semantic_results = evaluate_semantic_review(completed_review, review_key)
    display(semantic_results)


,theme,candidate_precision,control_precision,enrichment,candidate_mean_relevance,control_mean_relevance,n_candidates_reviewed,n_controls_reviewed,n_candidates_total,n_controls_total
0,Agribusiness,0.000000,0.000000,NaN,0.000000,0.100000,10,10,10,10
1,Clean Energy,0.300000,0.100000,3.0,1.200000,0.200000,10,10,10,10
2,Robotics & AI,0.000000,0.300000,0.0,0.200000,0.800000,10,10,10,10
3,Cybersecurity,0.000000,0.000000,NaN,0.700000,0.300000,10,10,10,10
4,Fintech,0.200000,0.000000,NaN,0.500000,0.200000,10,10,10,10
5,Water,0.500000,0.000000,NaN,1.500000,0.500000,2,2,2,2
6,Timber & Forestry,0.000000,0.000000,NaN,0.000000,0.100000,10,10,10,10
7,Critical Minerals,0.200000,0.000000,NaN,1.100000,0.000000,10,10,10,10
8,US Infrastructure,0.333333,0.333333,1.0,1.000000,1.000000,3,3,3,3
9,Defense,0.000000,0.000000,NaN,0.700000,0.200000,10,10,10,10


## Interpretation

A credible theme should show positive forward rank correlation, a positive top versus bottom exposure spread, positive basket correlation to the residualized ETF blend, and a low placebo p value. The semantic review is a separate economic check. Raw basket return remains secondary because the objective is exposure discovery rather than replication of a large cap ETF.

In [ ]:
# -------------------- 8. Export run results for review --------------------
from pathlib import Path
import zipfile

repo = Path.cwd()
outputs_dir = repo / "outputs"
outputs_dir.mkdir(exist_ok=True)

# Export the new diagnostics automatically.
results["evaluation"].to_csv(outputs_dir / "v3_forward_evaluation.csv", index=False)
results["candidate_forward_exposure"].to_csv(
    outputs_dir / "v3_candidate_forward_exposure.csv",
    index=False,
)
results["placebo_rank_thresholds"].to_csv(
    outputs_dir / "v3_placebo_rank_thresholds.csv",
    index=False,
)
results["theme_reference_distances"].to_csv(
    outputs_dir / "v3_theme_reference_distances.csv",
    index=False,
)
results["overlap_diagnostics"].to_csv(
    outputs_dir / "v3_overlap_diagnostics.csv",
    index=False,
)
results["timings"].to_csv(outputs_dir / "v3_timings.csv", index=False)
basket_history.to_csv(outputs_dir / "v3_basket_history.csv", index=False)

zip_path = repo / "ThemeCloner_V3_run_results.zip"
files_to_include = [
    repo / "ThemeCloner_V2_WalkForward_revised.ipynb",
    repo / "src" / "matching_v3.py",
    repo / "src" / "walkforward_v3.py",
    repo / "src" / "evaluation_v3.py",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_include:
        if file.exists():
            z.write(file, file.relative_to(repo))

    for file in outputs_dir.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(repo))

print(f"Created: {zip_path}")


In [ ]:
# -------------------- 9. Git commit and push --------------------
# Save the notebook before running this cell.
import subprocess

FILES_TO_COMMIT = [
    "ThemeCloner_V2_WalkForward_revised.ipynb",
    "src/matching_v3.py",
    "src/walkforward_v3.py",
    "src/evaluation_v3.py",
]

def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=ROOT,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result

run_git("status", "--short")
run_git("add", *FILES_TO_COMMIT)
run_git("diff", "--cached", "--stat")
run_git("commit", "-m", "Improve theme matching diagnostics and parallel placebos")
run_git("push")
